<a href="https://colab.research.google.com/github/Ololade117/stat4datascience/blob/main/SQ4DS_Lecture_3_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Statistics and Quantitative Methods for Data Science
[© Silvadew Institute](https://institute.silvadew.com/course-details/?c=43177d50)



# Lab: Building and Testing an N-gram Language Model

## What you will discover

In this lab, you will build a small **N-gram language model** from scratch.

You will start with a very simple model and gradually give it more context:

- **N = 1** → unigram model
- **N = 2** → bigram model
- **N = 3** → trigram model
- then you will investigate **N = 4, 5, and 6**

The important question is **not** "What is the largest N I can use?"

Instead, you will use a test set and a metric called **perplexity** to investigate:

> **Does giving a language model more context always make it better?**

### Lab rules

This is a **build-it-yourself lab**. Several code cells intentionally contain `TODO` sections or starter variables. Read the instructions, write the missing code, run the cell, and inspect the result before moving on.

By the end, you should be able to explain why a higher-order N-gram model can become less useful when the training data is limited.



## Learning objectives

By the end of the lab, you should be able to:

1. Explain what an N-gram is.
2. Create N-grams from a sequence of tokens.
3. Count N-grams and the contexts that precede them.
4. Estimate next-word probabilities from N-gram counts.
5. Generate a simple next-word prediction.
6. Evaluate an N-gram model using **perplexity**.
7. Compare several values of N and explain the effect of **sparse data**.


In [ ]:

# Run this cell first.
import re
import math
from collections import Counter
import matplotlib.pyplot as plt

print("Setup complete.")



## 1. Our mini training corpus

We will use a small, self-contained corpus so that the lab works without downloading anything from the internet.

The corpus is intentionally small. That is useful here because a small corpus makes the **data sparsity problem** easier to see.


In [ ]:
sentences = ['The student reads the data carefully.', 'The student cleans the data carefully.', 'The student splits the data into training and testing sets.', 'The student trains the model on the training data.', 'The student tests the model on the testing data.', 'The model predicts the next word from the context.', 'The model learns patterns from the training data.', 'The model uses word counts to estimate probability.', 'The model can memorize rare sequences in the training data.', 'The model may perform poorly on unseen sequences.', 'A unigram model ignores word order.', 'A bigram model uses one previous word.', 'A trigram model uses two previous words.', 'A higher order model uses more context.', 'More context can help when the context is common.', 'More context can hurt when the context is rare.', 'Students compare models on the same test set.', 'Students record perplexity for each model.', 'Students visualize perplexity with a simple chart.', 'A lower perplexity means the model assigns more probability to the test data.', 'The best ngram size depends on the amount of training data.', 'Sparse data makes higher order ngrams harder to estimate.', 'Rare contexts create unreliable probability estimates.', 'Good models balance context and data.', 'The lab asks students to try several ngram sizes.', 'Students try n equal to four five and six.', 'Students do not assume that larger n means a better model.', 'The final model should be chosen using evidence from the test set.']

print(f"Number of sentences: {len(sentences)}")
print("First sentence:", sentences[0])


### Why split the data?

A language model can look excellent if we only ask it about sentences it has already seen.

We therefore split the corpus into:

- a **training set**: used to learn N-gram counts
- a **test set**: kept separate so we can measure how well the model handles new text

For this lab, the split is fixed so everyone works with the same experiment.


In [ ]:

# Use the first 22 sentences for training and the remaining sentences for testing.
train_sentences = sentences[:22]
test_sentences = sentences[22:]

print("Training sentences:", len(train_sentences))
print("Test sentences:", len(test_sentences))
print()
print("Example test sentence:", test_sentences[0])



## 2. Tokenization

Before we can make N-grams, we need a sequence of tokens.

For this lab, use a simple tokenizer that:

1. converts text to lowercase
2. keeps alphabetic word tokens
3. returns the tokens as a Python list

### Your task

Complete the function below.

**Hint:** `re.findall()` can find all word-like pieces of text.


In [ ]:

def tokenize(text):
    # TODO 1:
    # Convert text to lowercase.
    # TODO 2:
    # Use re.findall(...) to return the word tokens.
    tokens = []
    return tokens


# Check your function after you complete it.
example = "The Model predicts the NEXT word!"
print(tokenize(example))



<details>
<summary><b>Hint</b></summary>

You can begin with:

```python
text = text.lower()
```

Then use a regular expression such as:

```python
re.findall(r"[a-z]+", text)
```

</details>



## 3. Create the tokenized corpus

Now tokenize every sentence.

We will store the result as a list of lists:

```text
[
    ["the", "student", "reads", ...],
    ["the", "student", "cleans", ...],
    ...
]
```


In [ ]:

train_tokens = [tokenize(sentence) for sentence in train_sentences]
test_tokens = [tokenize(sentence) for sentence in test_sentences]

print("First training sentence as tokens:")
print(train_tokens[0])



## 4. What is an N-gram?

An **N-gram** is a sequence of `N` consecutive tokens.

For example, suppose the sentence is:

```text
the student reads data
```

Then:

- N = 1 → `the`, `student`, `reads`, `data`
- N = 2 → `the student`, `student reads`, `reads data`
- N = 3 → `the student reads`, `student reads data`

### Your task

Write a function `make_ngrams(tokens, n)` that returns a list of tuples.

For example:

```python
make_ngrams(["the", "student", "reads"], 2)
```

should produce:

```python
[("the", "student"), ("student", "reads")]
```


In [ ]:

def make_ngrams(tokens, n):
    # TODO:
    # Return every consecutive n-token sequence as a tuple.
    #
    # Think carefully about:
    #   - the first valid starting position
    #   - the last valid starting position
    ngrams = []
    return ngrams


# Test your function.
demo_tokens = ["the", "student", "reads", "data"]
print("Bigrams:", make_ngrams(demo_tokens, 2))
print("Trigrams:", make_ngrams(demo_tokens, 3))



<details>
<summary><b>Hint</b></summary>

For a list of length `L`, there are:

```text
L - n + 1
```

possible N-grams.

A loop over `range(L - n + 1)` is a useful starting point.
</details>



## 5. Counting N-grams

The simplest language model idea is:

> **Count how often sequences occur, then use those counts to estimate probabilities.**

We will count both:

- the complete N-grams
- the `(N-1)`-word **contexts**

For a bigram model:

```text
context = ("the",)
ngram   = ("the", "student")
```

For a trigram model:

```text
context = ("the", "student")
ngram   = ("the", "student", "reads")
```

### Your task

Complete the function below.

Return:

```python
ngram_counts, context_counts
```

### Boundary markers

To let us predict the first word of a sentence too, the function should add `"<s>"` tokens to the **left side** of every sentence:

```text
N=3
["<s>", "<s>", "the", "student", "reads", ...]
```

This gives the first real word a full two-word context.


In [ ]:

def count_ngrams(tokenized_sentences, n):
    # TODO:
    # 1. Create a Counter for complete N-grams.
    # 2. Create a Counter for (N-1)-token contexts.
    # 3. For every sentence, add (n-1) "<s>" markers to the left.
    # 4. Build the N-grams using make_ngrams(...).
    # 5. Update both counters.
    ngram_counts = Counter()
    context_counts = Counter()

    return ngram_counts, context_counts


bigram_counts, bigram_contexts = count_ngrams(train_tokens, 2)

print("Most common bigrams:")
for gram, count in bigram_counts.most_common(10):
    print(gram, "->", count)



### Inspect one context

Once your counter works, run this cell.

What happens to the counts when a context appears many times? What happens when a context appears only once?


In [ ]:

context = ("the",)
print("Count of context", context, "=", bigram_contexts[context])

print("\nBigrams beginning with 'the':")
for gram, count in bigram_counts.items():
    if gram[0] == "the":
        print(gram, "->", count)



## 6. From counts to probabilities

For a bigram model, the probability of the next word can be estimated as:

\[
P(w_t \mid w_{t-1}) =
\frac{\text{count}(w_{t-1}, w_t)}
{\text{count}(w_{t-1})}
\]

For a general N-gram model:

\[
P(w_t \mid context) =
\frac{\text{count(context + }w_t\text{)}}
{\text{count(context)}}
\]

### But there is a problem...

Suppose a context never occurred in the training data.

Then its count is zero, and an unseen N-gram gets probability zero.

Zero probabilities are a serious problem when we evaluate a whole sentence.

So we will use **add-k smoothing**:

\[
P =
\frac{\text{count(ngram)} + k}
{\text{count(context)} + kV}
\]

where:

- `k` is a small smoothing value
- `V` is the vocabulary size

We will use `k = 0.1` for the main experiment.



## 7. Build a next-word predictor

Your model should take a context and return the most probable next word.

Example:

```text
context = ("the", "student")
N = 3
```

The function should look at all trigrams beginning with that context and choose a next word using the smoothed probability.

### Your task

Complete `predict_next_word()`.


In [ ]:

def predict_next_word(context, ngram_counts, context_counts, vocabulary, k=0.1):
    # context should contain n-1 words.
    #
    # TODO:
    # 1. Find candidate next words from ngram_counts.
    # 2. Compute the smoothed probability for each candidate.
    # 3. Return the candidate with the highest probability.
    #
    # Important:
    # A useful first step is to inspect all ngrams whose
    # first n-1 tokens equal the supplied context.
    best_word = None
    best_probability = -1

    return best_word


# Start with a trigram model.
trigram_counts, trigram_contexts = count_ngrams(train_tokens, 3)
vocabulary = sorted(set(word for sentence in train_tokens for word in sentence))

context = ("the", "student")
print("Predicted next word:", predict_next_word(
    context, trigram_counts, trigram_contexts, vocabulary
))



<details>
<summary><b>Hint</b></summary>

For each candidate `word`, the full N-gram is:

```python
context + (word,)
```

The smoothed probability is:

```python
(ngram_counts[full_ngram] + k) / (
    context_counts[context] + k * len(vocabulary)
)
```

The candidate with the highest probability is the prediction.

</details>



## 8. Generate a short sequence

Now combine your predictor with a loop.

Starting from a short context, repeatedly predict the next word and append it.

This is a tiny text generator—not a modern neural language model. The purpose is to make the N-gram mechanism visible.


In [ ]:

def generate_text(start_context, ngram_counts, context_counts, vocabulary,
                  steps=8):
    words = list(start_context)

    # TODO:
    # Repeat `steps` times:
    #   1. Take the last n-1 words as the current context.
    #   2. Predict the next word.
    #   3. If no word can be predicted, stop.
    #   4. Append the prediction to `words`.
    #
    # Return the generated words as a single string.

    return " ".join(words)


start = ("the", "student")
print(generate_text(
    start,
    trigram_counts,
    trigram_contexts,
    vocabulary,
    steps=8
))



## 9. Evaluate the model with perplexity

A model that assigns high probability to the test data should have **lower perplexity**.

Conceptually:

\[
PP =
\exp\left(
-\frac{1}{T}\sum_{t=1}^{T}\log P(w_t \mid context_t)
\right)
\]

You do not need to memorize the formula yet. Focus on the idea:

> **Lower perplexity means the model assigned more probability to the observed test words.**

### Your task

Complete `perplexity()`.

Requirements:

1. Build the N-gram counts from the **training** data.
2. Use the test data only for evaluation.
3. Add `(N-1)` `"<s>"` tokens to the left of every test sentence.
4. Evaluate every real word in the test sentence.
5. Use add-k smoothing.
6. Compute the average negative log probability.
7. Return the exponential of that average.


In [ ]:

def perplexity(train_tokenized, test_tokenized, n, k=0.1):
    ngram_counts, context_counts = count_ngrams(train_tokenized, n)
    vocabulary = set(word for sentence in train_tokenized for word in sentence)
    V = len(vocabulary)

    total_log_probability = 0.0
    total_predictions = 0

    # TODO:
    # For each test sentence:
    #   1. Add (n-1) "<s>" markers to the left.
    #   2. Loop through each real test word.
    #   3. Build the n-1 word context.
    #   4. Build the full ngram.
    #   5. Compute the smoothed probability.
    #   6. Add math.log(probability) to total_log_probability.
    #   7. Increment total_predictions.
    #
    # Finally:
    # average_negative_log_probability = - total_log_probability / total_predictions
    # return math.exp(average_negative_log_probability)

    return None


# Run after completing the function.
print("Bigram perplexity:", perplexity(train_tokens, test_tokens, 2))



<details>
<summary><b>Hint</b></summary>

For a test sentence such as:

```text
the model predicts the next word
```

and `N = 3`, pad the sentence as:

```text
<s> <s> the model predicts the next word
```

The prediction for `"the"` uses:

```text
context = ("<s>", "<s>")
target  = "the"
```

The prediction for `"predicts"` uses:

```text
context = ("the", "model")
target  = "predicts"
```

A smoothed probability can be computed even when an N-gram was unseen in training.
</details>



# 10. First experiment: N = 1, 2, 3

Now we stop building the model one piece at a time and start behaving like experimentalists.

We will compare several model sizes using the **same training set**, **same test set**, and **same smoothing value**.

### Your task

Complete the loop so it calculates perplexity for:

```text
N = 1, 2, 3
```

Then print the results.


In [ ]:

results = {}

# TODO:
# Loop through n values 1, 2, and 3.
# Store each perplexity in `results[n]`.

print(results)



### Think before you continue

Look at your first results.

Do not decide that "bigger N is better" just because a model uses more context.

Ask:

- Did perplexity decrease every time?
- What happened when the model moved from N=2 to N=3?
- How much training data does a trigram need compared with a unigram?



# 11. The main experiment: try N = 4, 5, and 6

This is the key part of the lab.

Higher-order models can represent longer contexts, but longer contexts are also **harder to observe repeatedly**.

For example, this sequence:

```text
"the student reads the"
```

is much more specific than:

```text
"the"
```

A very specific context may appear once—or not at all.

That creates **sparsity**.

### Your task

Run a new experiment using:

```text
N = 4, 5, 6
```

Store the perplexities in the same `results` dictionary.

Then print every N and its perplexity in order.


In [ ]:

# TODO:
# Extend the experiment to N = 4, 5, and 6.
# Keep the existing values in `results`.
#
# Example target structure:
# results = {1: ..., 2: ..., 3: ..., 4: ..., 5: ..., 6: ...}


for n in sorted(results):
    print(f"N={n}: perplexity={results[n]:.2f}")



## 12. Visualize the results

A graph can make the trade-off easier to see.

### Your task

Create a line plot with:

- x-axis → `N`
- y-axis → perplexity
- one point for each tested N

Add axis labels and a title.

After plotting, describe the overall shape of the curve in one or two sentences.


In [ ]:

# TODO:
# Create a line plot from the `results` dictionary.
#
# Useful building blocks:
# x = sorted(results)
# y = [results[n] for n in x]
# plt.plot(x, y, marker="o")
# plt.xlabel(...)
# plt.ylabel(...)
# plt.title(...)
# plt.show()




# 13. Zoom in on N = 4, 5, 6

The lab specifically asks you to try several larger N values.

### Your task

Create a small table-like printout showing only:

```text
N = 4
N = 5
N = 6
```

Then answer the questions below in the notebook.

1. Which of N=4, 5, and 6 produced the lowest perplexity **on this test set**?
2. Does increasing N from 4 → 5 → 6 consistently improve the score?
3. What does that tell you about the phrase **"more context"**?
4. Why might a context that is too specific be difficult to estimate reliably?


In [ ]:

# TODO:
# Print the results for N = 4, 5, and 6.
for n in [4, 5, 6]:
    print(f"N={n}: {results[n]:.2f}")



# 14. See the sparsity directly

Perplexity tells us **what happened**. Now we will inspect **why**.

For each N, count how many distinct N-grams were observed in the training data.

### Your task

For N = 1 through 6:

1. build the N-gram counts
2. print the number of distinct N-grams

Then compare those counts with the original number of training sentences.

What pattern do you notice?

> **Important:** The number of distinct N-grams may increase as N increases, while the number of times many individual contexts repeat may stay low. That is one reason high-order N-grams become sparse.


In [ ]:

ngram_type_counts = {}

# TODO:
# For n in 1 through 6:
#   1. Build the N-gram counts.
#   2. Store len(counts) in ngram_type_counts[n].

for n in range(1, 7):
    print(f"N={n}: {ngram_type_counts.get(n, 'TODO')}")



# 15. Final reflection

Complete these statements in your own words.

### A. What does N control?

> "The value of N controls ____________________________________________."

### B. Why can a larger N be difficult?

> "As N increases, contexts become more specific, so __________________________."

### C. What did your experiment show?

> "On this dataset, increasing N did / did not consistently improve perplexity because __________________________."

### D. What is the practical lesson?

> "I should not automatically choose the largest N. Instead, I should __________________________."

### E. Model selection

Suppose a model with N=2 has lower test perplexity than N=6.

Why would choosing N=6 simply because it uses more context be a poor experimental decision?



# 16. Extension challenge

You have now seen one important limitation of N-grams: **sparsity**.

Try one of these experiments:

### Challenge 1 — More training data

Move a few sentences from the test set into the training set.

Run N=4, 5, and 6 again.

Does the behavior change?

### Challenge 2 — Change the smoothing value

Change `k` from `0.1` to another positive value.

Does the shape of the perplexity curve change?

### Challenge 3 — Try N=7

Add N=7 to the experiment.

Does the extra context help, hurt, or make little difference?

### Challenge 4 — Explain the trade-off

Write a short paragraph answering:

> **Why might a moderate N be a better choice than either a very small N or a very large N?**

---

## Key takeaway

The goal of this lab is **not** to find a magical "best N."

The goal is to discover experimentally that:

> **More context does not automatically mean a better language model.**

Higher-order N-grams can capture longer patterns, but they also create more specific contexts. With limited data, many of those contexts are rare, which makes their probabilities harder to estimate reliably.

That is why a model builder should **try several values of N and evaluate them on held-out data**, rather than assuming that the largest N must be best.
